In [ ]:
###############################################################################
# Cell 0 (필수) — 프로젝트 소스코드 업로드                                      #
#                                                                             #
# 최초 1회만 실행하면 됩니다. 이후 Drive에 저장된 code/를 재사용합니다.         #
# 만약 code/가 이미 Drive에 있으면 이 셀은 건너뛰고 Cell 3부터 실행하세요.       #
###############################################################################
from google.colab import files
uploaded = files.upload()  # ← moe_code.zip (또는 transformer_code.zip) 선택


In [ ]:
###############################################################################
# Cell 1 (필수) — Colab 환경 설정                                              #
#                                                                             #
# 1. PyTorch (CUDA 12.4) + accelerate + datasets + tokenizers + wandb 설치     #
# 2. A100 GPU 연결 확인                                                       #
# 3. Google Drive 마운트 (korean_chat 폴더)                                    #
# 4. Python path 및 디렉토리 생성                                              #
###############################################################################
!pip install torch --index-url https://download.pytorch.org/whl/cu124 -q --upgrade
!pip install accelerate datasets tokenizers wandb -q

import torch
assert torch.cuda.is_available()
print(f"✅ {torch.cuda.get_device_name()} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys, os
SRC = "/content/drive/MyDrive/korean_chat"
CKPT_DIR = f"{SRC}/checkpoints"
LOG_DIR = f"{SRC}/logs"
DATA_DIR = f"{SRC}/chat_data"
TOKENIZER_DIR = f"{SRC}/tokenizer"

# ★ 중요: train.py 등이 model/ 패키지를 import할 수 있도록 code/ 경로 추가
sys.path.insert(0, f"{SRC}/code")

for d in [CKPT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Cell 1 완료 — A100 + Drive 준비됨")


In [ ]:
###############################################################################
# Cell 2 (선택) — 소스코드 경로 확인                                           #
#                                                                             #
# 앞선 셀에서 설정한 경로와 sys.path가 올바른지 확인합니다.                     #
# 에러 없이 모든 항목이 ✅ 로 표시되면 정상입니다.                              #
###############################################################################
import sys, os
SRC = "/content/drive/MyDrive/korean_chat"
print("SRC exists:", os.path.exists(SRC))
print("code/ exists:", os.path.exists(f"{SRC}/code"))
print("model/ exists:", os.path.exists(f"{SRC}/code/model"))
print("dense_transformer.py:", os.path.exists(f"{SRC}/code/model/dense_transformer.py"))
print()
print("sys.path entries:")
for p in sys.path:
    if "drive" in p.lower() or "chat" in p.lower():
        print(f"  ✅ {p}")
    else:
        print(f"     {p}")


In [ ]:
###############################################################################
# Cell 3 (최초 1회) — 업로드한 ZIP → Drive로 복사                              #
#                                                                             #
# Cell 0에서 업로드한 ZIP 파일을 /content/project_code/에 풀고,                #
# 이를 Drive(korean_chat/code/)로 복사합니다.                                  #
# 이후 Cell 1~3을 다시 실행할 때는 code/가 이미 Drive에 있으므로 생략 가능.     #
###############################################################################
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
import shutil

SRC = "/content/drive/MyDrive/korean_chat"

zip_filename = "moe_code.zip"
if os.path.exists(zip_filename):
    print(f"📦 Extracting {zip_filename}...")
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall("/content/project_code")

    # Drive code/ 폴더 갱신 (기존 내용 삭제 후 복사)
    if os.path.exists(f"{SRC}/code"):
        shutil.rmtree(f"{SRC}/code")
    shutil.copytree("/content/project_code", f"{SRC}/code")
    print(f"✅ Code copied to {SRC}/code")
else:
    print("⚠️ ZIP 파일 없음 — 이미 Drive에 code/가 있다면 건너뛰어도 됩니다.")

# 하위 디렉토리 생성
for d in ["checkpoints", "logs", "tokenizer", "chat_data", "logs/events"]:
    os.makedirs(f"{SRC}/{d}", exist_ok=True)
print(f"✅ Directories ready at {SRC}")


In [ ]:
###############################################################################
# Cell 4 (필수) — AI Hub 데이터 압축 해제 + 한국어 BPE 토크나이저 학습          #
#                                                                             #
# 1. Drive에 업로드된 TL_session*.zip 파일들을 Colab 로컬 디스크에 압축 해제     #
#    (Drive FUSE I/O 병목 회피 → 로컬 SSD 사용으로 속도 10배 향상)              #
# 2. 데이터로 BPE 토크나이저 학습 (vocab_size 32,000)                          #
#                                                                             #
# ⚠️ AI Hub 데이터가 없으면 --data_dir 경로를 비워두거나 실제 경로로 수정       #
###############################################################################
import os
import glob
import zipfile

SRC = "/content/drive/MyDrive/korean_chat"
AIHUB_ZIP_DIR = "/content/drive/MyDrive/aihub_data"      # 🗂️ TL_session*.zip 파일 위치
EXTRACT_DIR = "/content/aihub_extracted"                  # 🚀 Colab 로컬 디스크 (고속)

# 1. Drive → Colab 로컬 디스크로 압축 해제
if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    zip_files = glob.glob(os.path.join(AIHUB_ZIP_DIR, "TL_session*.zip"))
    if not zip_files:
        print(f"⚠️ {AIHUB_ZIP_DIR} 에 TL_session*.zip 없음 — HF 데이터만으로 진행")
    for zip_path in zip_files:
        print(f"📦 Extracting {os.path.basename(zip_path)}...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(EXTRACT_DIR)
        except Exception as e:
            print(f"❌ {zip_path} 실패: {e}")
    print("✅ Extraction complete!")
else:
    print(f"✅ 이미 압축 해제됨: {EXTRACT_DIR}")

# 2. BPE 토크나이저 학습
!python {SRC}/code/tokenizer/train_tokenizer.py \
    --korean \
    --data_dir {EXTRACT_DIR} \
    --output_dir {SRC}/tokenizer


In [ ]:
###############################################################################
# Cell 5 (필수) — 데이터 전처리                                                 #
#                                                                             #
# AI Hub 데이터 + HuggingFace 한국어 데이터셋 7종을 통합하여:                   #
#   <user>{질문}<sep><assistant>{답변}</s>                                    #
# 포맷으로 토크나이징, block_size=512 청크로 분할, train/val 9:1 분할           #
#                                                                             #
# 결과: {SRC}/chat_data/train.npy + val.npy                                     #
###############################################################################
import os

SRC = "/content/drive/MyDrive/korean_chat"
EXTRACT_DIR = "/content/aihub_extracted"

!python {SRC}/code/train/prepare_chat_data.py \
    --data_dir {EXTRACT_DIR} \
    --hf_datasets beomi/KoAlpaca-v1.1a nlpai-lab/kullm-v2 beomi/KoAlpaca-RealQA \
                   kyujinpy/KOR-OpenOrca-Platypus-v3 JaeJiMin/korean_chat_friendly \
                   FreedomIntelligence/sharegpt-korean BAEM1N/nanochat_korean \
    --tokenizer_dir {SRC}/tokenizer \
    --output_dir {SRC}/chat_data \
    --block_size 512


In [ ]:
###############################################################################
# Cell 6 (필수) — Dense Transformer Fine-Tuning (v5)                           #
#                                                                             #
# 162M Dense Transformer를 한국어 챗봇 데이터로 추가 학습합니다.               #
#                                                                             #
# v5 변경점:                                                                  #
#   • epochs 1 → 3  (loss 2,250step 기준 4.2로 아직 수렴 전 = 추가 학습 필요)  #
#   • warmup 500 → 200 (1epoch 대비 20%→8%, 실제 학습 구간 확보)              #
#   • dropout 0.1 → 0.15 (3epoch 과적합 방지)                                 #
#   • run_id: korean_chat_v5                                                  #
#                                                                             #
# ⚠️ 사전 준비:                                                               #
#   - 사전학습된 Dense Transformer 체크포인트(moe-to-dense 변환본) 필요          #
#   - checkpoint_manager.py를 통해 resume 기능 자동 활성화                     #
###############################################################################
import os
# import wandb  # wandb 로깅 활성화 시 주석 해제
# wandb.login()

SRC = "/content/drive/MyDrive/korean_chat"

!PYTHONPATH={SRC}/code python -m train.train \
    --run_id korean_chat_v5 \
    --name korean_chat_model_v3 \
    --data_dir {SRC}/chat_data \
    --tokenizer_dir {SRC}/tokenizer \
    --project_dir {SRC} \
    --batch_size 32 \
    --block_size 512 \
    --lr 3e-4 \
    --epochs 3 \
    --save_every 1000 \
    --log_every 100 \
    --warmup_steps 200 \
    --dropout 0.15 \
    --wandb


---
### ✅ 학습 완료 후 아래 셀들을 실행하여 결과를 확인하세요
---


In [ ]:
###############################################################################
# Cell 8 (선택) — 학습된 모델 추론(Inference) 테스트                            #
#                                                                             #
# v5 학습이 완료된 최종 체크포인트를 불러와 대화 생성 테스트를 수행합니다.      #
# --chat 플래그가 자동으로 <user>프롬프트<sep><assistant> 포맷으로 변환합니다. #
#                                                                             #
# ⚠️ 체크포인트 경로는 학습 완료 후 실제 저장된 파일명으로 수정 필요            #
#    (ls {SRC}/checkpoints/dense_korean_chat_v5_* 로 확인)                     #
###############################################################################
import os

SRC = "/content/drive/MyDrive/korean_chat"

# 🎯 학습 완료 후 아래 경로를 실제 체크포인트 파일명으로 변경하세요
#    예) dense_korean_chat_v5_step7305.pt (3epoch × 2,435step)
checkpoint_path = f"{SRC}/checkpoints/dense_korean_chat_v5.pt"

!python {SRC}/code/train/generate.py \
    --checkpoint {checkpoint_path} \
    --tokenizer_dir {SRC}/tokenizer \
    --prompt "안녕하세요? 오늘 날씨가 좋네요" \
    --chat


In [ ]:
###############################################################################
# Cell 9 (선택) — 로컬 배포용 체크포인트 변환                                   #
#                                                                             #
# Colab A100에서 BF16 DDP로 학습된 체크포인트를                                #
# 로컬(Mac MPS/CPU) 추론용 FP32 단일 파일로 변환합니다.                        #
#                                                                             #
# 변환 결과: {SRC}/export/korean_chat.pt (→ serve/app.py에서 로드)              #
###############################################################################
import os

SRC = "/content/drive/MyDrive/korean_chat"

# 🎯 변환할 소스 체크포인트 (Cell 8과 동일한 파일 지정)
src_ckpt = f"{SRC}/checkpoints/dense_korean_chat_v5.pt"
dst_ckpt = f"{SRC}/export/korean_chat.pt"

!python {SRC}/code/train/export_for_local.py \
    --src {src_ckpt} \
    --dst {dst_ckpt}
